# 09 — Human Evaluation of Explanation Quality

**Purpose:** Evaluate 50 BGL + 50 HDFS explanations on four dimensions via manual review.

**Rubric (Likert 1-5):**

| Dimension | 1 | 3 | 5 |
|-----------|---|---|---|
| **Correctness** | Explanation contradicts the logs | Partially correct, minor errors | Fully accurate description of the anomaly |
| **Completeness** | Major anomaly signals missing | Covers main signal, misses secondary | All anomaly signals addressed |
| **Evidence Grounding** | Claims cite wrong or no evidence | Most claims grounded, some vague | Every claim backed by correct evidence span |

**Binary:**
| Dimension | Y | N |
|-----------|---|---|
| **Actionable** | An engineer could act on this explanation | Too vague or wrong to act on |

**Workflow:**
1. Run Sections 1-3 once (Section 2 is slow — loads raw logs, cached after first run)
2. Section 4: evaluating — change `IDX`, view session, fill in `rate()`, run
3. Section 5: progress check
4. Section 6: analysis (run after all 100 sessions rated)

In [1]:
import sys, json, math, random
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# === Constants ===
SAMPLE_SIZE = 50
SEED = 42

RESULT_FILES = {
    'BGL':  PROJECT_ROOT / 'results' / 'explanations_BGL_20260213_160034.normalized.jsonl',
    'HDFS': PROJECT_ROOT / 'results_HDFS' / 'explanations_HDFS_20260215_220648.normalized.jsonl',
}

PREPARED_PATH = PROJECT_ROOT / 'results' / 'human_eval_prepared.json'
RATINGS_PATH  = PROJECT_ROOT / 'results' / 'human_eval_ratings.json'

print(f"Project root: {PROJECT_ROOT}")
print(f"[OK] Imports loaded")

Project root: /home/dave/agentic-log-explanations
[OK] Imports loaded


## 1. Stratified Sampling

Sample 50 sessions per dataset, stratified by signature to ensure coverage.

In [2]:
# === Stratified sampling from normalized JSONL ===

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if d.get('explanation') is not None:
                records.append(d)
    return records


def get_sig(record):
    return record.get('normalized_signature',
                      record['explanation'].get('signature', {}).get('name', 'UNKNOWN'))


def stratified_sample(records, n, seed):
    """Stratified random sample ensuring max signature coverage."""
    rng = random.Random(seed)
    by_sig = {}
    for r in records:
        by_sig.setdefault(get_sig(r), []).append(r)

    sigs_sorted = sorted(by_sig.keys(), key=lambda s: len(by_sig[s]), reverse=True)

    sampled, used_ids = [], set()
    budget = n

    # Phase 1: one per signature
    for sig in sigs_sorted:
        if budget <= 0:
            break
        pick = rng.choice(by_sig[sig])
        sampled.append(pick)
        used_ids.add(pick['session_id'])
        budget -= 1

    # Phase 2: proportional fill
    if budget > 0:
        pool = [r for sig in sigs_sorted for r in by_sig[sig]
                if r['session_id'] not in used_ids]
        rng.shuffle(pool)
        sampled.extend(pool[:budget])

    rng.shuffle(sampled)
    return sampled


samples = {}
for ds_name, path in RESULT_FILES.items():
    records = load_jsonl(path)
    sampled = stratified_sample(records, SAMPLE_SIZE, SEED)
    samples[ds_name] = sampled

    sigs = Counter(get_sig(r) for r in sampled)
    print(f"[OK] {ds_name}: sampled {len(sampled)} / {len(records)} sessions, "
          f"{len(sigs)} signatures covered")
    for sig, cnt in sigs.most_common(5):
        print(f"     {sig}: {cnt}")
    print()

[OK] BGL: sampled 50 / 6290 sessions, 50 signatures covered
     KERNEL__MICROLOADER_ASSERTION: 1
     MMCS__ASSERT_CONDITION: 1
     KERNEL__INSTRUCTION_ADDRESS: 1
     APP__CIOD_NODE_MAP_ERROR: 1
     KERNEL__INTEGER_ALIGNMENT_ERROR: 1

[OK] HDFS: sampled 50 / 2527 sessions, 13 signatures covered
     DATANODE__BLOCK_VERIFICATION_FAILED: 25
     DATANODE__BLOCK_WRITE_FAILURE: 10
     DATANODE__BLOCK_SERVING_FAILURE: 3
     DATANODE__BLOCK_TRANSFER_FAILURE: 2
     DATANODE__BLOCK_RECEIVE_FAILURE: 2



## 2. Load Log Lines & Prepare Data

**First run takes several minutes** (loads BGL 709 MB + HDFS 1.5 GB).
Results are cached to `human_eval_prepared.json` — subsequent runs skip loading.

In [3]:
# === Load raw log lines for sampled sessions (cached) ===

if PREPARED_PATH.exists():
    print(f"[SKIP] Prepared data already cached at {PREPARED_PATH.name}")
    print("       Delete the file and re-run this cell to regenerate.")
    with open(PREPARED_PATH) as f:
        prepared = json.load(f)
    for ds in prepared:
        print(f"       {ds}: {len(prepared[ds])} sessions")
else:
    from src.data_loader import BGLDataLoader, HDFSDataLoader

    sid_to_lines = {}

    # BGL
    print("Loading BGL dataset (~2 min)...")
    bgl_loader = BGLDataLoader(
        log_file=str(PROJECT_ROOT / 'logs' / 'BGL.log'),
        windows_size=10, step_size=10,
    ).load()
    for s in bgl_loader.get_sessions():
        sid_to_lines[s.session_id] = s.lines
    print(f"  {len([k for k in sid_to_lines if k.startswith('BGL')])} BGL sessions indexed")

    # HDFS
    print("Loading HDFS dataset (~5 min)...")
    hdfs_loader = HDFSDataLoader(
        log_file=str(PROJECT_ROOT / 'logs' / 'HDFS.log'),
        label_file=str(PROJECT_ROOT / 'logs' / 'anomaly_label_HDFS.csv'),
    ).load()
    for s in hdfs_loader.get_sessions():
        sid_to_lines[s.session_id] = s.lines
    print(f"  {len(sid_to_lines)} total sessions indexed")

    # Build prepared data
    prepared = {}
    missing = 0
    for ds_name, recs in samples.items():
        prepared[ds_name] = []
        for r in recs:
            sid = r['session_id']
            lines = sid_to_lines.get(sid)
            if lines is None:
                missing += 1
                lines = ['[LOG LINES NOT FOUND]']
            entry = {
                'session_id': sid,
                'dataset': ds_name,
                'normalized_signature': get_sig(r),
                'log_lines': lines,
                'margin': r['screener']['margin'],
                'verification_passed': r.get('verification_passed'),
                'verification_checks': r.get('verification_checks'),
                'verification_failed': r.get('verification_failed_checks'),
                'summary': r['explanation']['summary'],
                'claims': r['explanation']['claims'],
                'evidence_id_mapping': r.get('evidence_id_mapping', {}),
            }
            prepared[ds_name].append(entry)

    with open(PREPARED_PATH, 'w') as f:
        json.dump(prepared, f, indent=1, ensure_ascii=False)

    print(f"\n[OK] Saved to {PREPARED_PATH.name}")
    if missing:
        print(f"[WARN] {missing} sessions had missing log lines")
    for ds in prepared:
        print(f"     {ds}: {len(prepared[ds])} sessions")

Loading BGL dataset (~2 min)...
Loading BGL logs from: /home/dave/agentic-log-explanations/logs/BGL.log


Reading BGL logs: 4747963it [00:01, 3474453.53it/s]


Loaded 4747963 log lines


Creating sessions: 100%|██████████| 474796/474796 [00:02<00:00, 193643.68it/s]


  474796 BGL sessions indexed
Loading HDFS dataset (~5 min)...
Loading HDFS logs from: /home/dave/agentic-log-explanations/logs/HDFS.log
Loading labels from: /home/dave/agentic-log-explanations/logs/anomaly_label_HDFS.csv


Reading HDFS logs: 11175629it [00:09, 1173775.21it/s]


Found 575061 unique blocks
  1049857 total sessions indexed

[OK] Saved to human_eval_prepared.json
     BGL: 50 sessions
     HDFS: 50 sessions


## 3. Evaluation Interface

Run this cell once to load functions and current progress.
Safe to re-run after kernel restart — ratings are saved to disk.

In [4]:
# === Evaluation: display + rate + save ===

# Load or initialize ratings
if RATINGS_PATH.exists():
    with open(RATINGS_PATH) as f:
        ratings_data = json.load(f)
    print(f"[OK] Loaded existing ratings from {RATINGS_PATH.name}")
else:
    ratings_data = {
        'evaluator': 'dave',
        'rubric_version': 'v1',
        'sample_size': SAMPLE_SIZE,
        'seed': SEED,
        'dimensions': {
            'correctness': '1-5: Is the explanation factually correct?',
            'completeness': '1-5: Does it cover all anomaly signals in the logs?',
            'evidence_grounding': '1-5: Are claims properly supported by cited evidence?',
            'actionable': 'Y/N: Could an engineer act on this explanation?',
        },
        'ratings': {},
    }
    print("[OK] Initialized empty ratings")


def _progress():
    for ds in ['BGL', 'HDFS']:
        n = sum(1 for s in prepared.get(ds, [])
                if s['session_id'] in ratings_data['ratings'])
        print(f"  {ds}: {n}/{len(prepared.get(ds, []))}")


def display_session(dataset, idx):
    """Display one session for evaluation."""
    sessions = prepared[dataset]
    assert 0 <= idx < len(sessions), f"IDX must be 0-{len(sessions)-1}"
    s = sessions[idx]
    sid = s['session_id']
    rated = '[RATED]' if sid in ratings_data['ratings'] else '[UNRATED]'

    W = 80
    print('=' * W)
    print(f"[{idx+1}/{len(sessions)}] {sid}  {rated}")
    print(f"Dataset: {dataset} | Signature: {s['normalized_signature']}")
    vp   = s.get('verification_passed')
    vc   = s.get('verification_checks') or 0
    vf   = s.get('verification_failed') or 0
    vstr = f"PASSED ({vc-vf}/{vc})" if vp else f"FAILED ({vf} issues)"
    print(f"Verification: {vstr} | Margin: {s['margin']:.6f}")
    print('=' * W)

    # Log lines
    print("\nLOG LINES (E0):")
    print('-' * W)
    for i, line in enumerate(s['log_lines'][:30]):
        print(f"  L{i+1:02d}: {line}")
    if len(s['log_lines']) > 30:
        print(f"  ... ({len(s['log_lines'])} lines total, showing first 30)")

    # Explanation
    print('\n' + '-' * W)
    print("EXPLANATION:")
    print(f"  Summary: {s['summary']}")
    print(f"\n  Claims:")
    for i, c in enumerate(s['claims']):
        ctype = c.get('type', '?')
        claim = c.get('claim', '')
        eids  = ', '.join(c.get('evidence_ids', []))
        spans = ', '.join(c.get('evidence_spans', []))
        print(f"\n  {i+1}. [{ctype}]")
        print(f"     {claim}")
        if eids:
            print(f"     Evidence: {eids} | Spans: {spans}")

    # Evidence mapping
    mapping = s.get('evidence_id_mapping', {})
    if mapping:
        print(f"\n  Evidence ID mapping:")
        for alias, real_id in mapping.items():
            print(f"    {alias} -> {real_id}")

    print('\n' + '=' * W)

    # Next unrated hint
    unrated = [i for i, ss in enumerate(sessions)
               if ss['session_id'] not in ratings_data['ratings']]
    if unrated and unrated[0] != idx:
        print(f"Next unrated: IDX = {unrated[0]}")


def rate(dataset, idx, correctness, completeness, evidence_grounding,
         actionable, notes=''):
    """Rate a session and save to disk."""
    sid = prepared[dataset][idx]['session_id']

    for name, val in [('correctness', correctness),
                      ('completeness', completeness),
                      ('evidence_grounding', evidence_grounding)]:
        assert isinstance(val, int) and 1 <= val <= 5, \
            f"{name} must be int 1-5, got {val}"
    assert actionable in ('Y', 'N'), f"actionable must be 'Y' or 'N', got {actionable!r}"

    ratings_data['ratings'][sid] = {
        'dataset': dataset,
        'idx': idx,
        'normalized_signature': prepared[dataset][idx]['normalized_signature'],
        'correctness': correctness,
        'completeness': completeness,
        'evidence_grounding': evidence_grounding,
        'actionable': actionable,
        'notes': notes,
        'rated_at': datetime.now().isoformat(),
    }

    with open(RATINGS_PATH, 'w') as f:
        json.dump(ratings_data, f, indent=2, ensure_ascii=False)

    n = sum(1 for s in prepared[dataset]
            if s['session_id'] in ratings_data['ratings'])
    print(f"[OK] {sid}: C={correctness} Co={completeness} "
          f"E={evidence_grounding} A={actionable}")
    print(f"     {dataset} progress: {n}/{len(prepared[dataset])}")


print("\nCurrent progress:")
_progress()
print("\n[OK] display_session() and rate() ready")

[OK] Initialized empty ratings

Current progress:
  BGL: 0/50
  HDFS: 0/50

[OK] display_session() and rate() ready


## 4. Evaluate

**How to use:**
1. Set `DATASET` and `IDX` in the **display cell** below, run it
2. Read the log lines and explanation
3. Fill in scores in the **rate cell**, run it
4. Increment `IDX` and repeat

Rating is auto-saved after each `rate()` call. You can stop and resume anytime.

In [5]:
# === DISPLAY: change DATASET and IDX, then run ===
DATASET = 'BGL'   # 'BGL' or 'HDFS'
IDX = 0           # 0-49

display_session(DATASET, IDX)

[1/50] BGL_03298600  [UNRATED]
Dataset: BGL | Signature: KERNEL__MICROLOADER_ASSERTION
Verification: PASSED (8/8) | Margin: 1.000000

LOG LINES (E0):
--------------------------------------------------------------------------------
  L01: 1125326832 2005.08.29 R51-M1-N5-C:J14-U01 2005-08-29-07.47.12.353424 R51-M1-N5-C:J14-U01 RAS KERNEL FATAL Microloader Assertion
  L02: 1125326832 2005.08.29 R51-M1-N5-C:J16-U01 2005-08-29-07.47.12.445971 R51-M1-N5-C:J16-U01 RAS KERNEL FATAL Microloader Assertion
  L03: 1125326832 2005.08.29 R51-M1-N5-C:J10-U01 2005-08-29-07.47.12.547491 R51-M1-N5-C:J10-U01 RAS KERNEL FATAL Microloader Assertion
  L04: 1125326832 2005.08.29 R51-M1-N5-C:J12-U01 2005-08-29-07.47.12.642367 R51-M1-N5-C:J12-U01 RAS KERNEL FATAL Microloader Assertion
  L05: 1125326832 2005.08.29 R51-M1-N5-C:J08-U01 2005-08-29-07.47.12.732055 R51-M1-N5-C:J08-U01 RAS KERNEL FATAL Microloader Assertion
  L06: 1125326832 2005.08.29 R51-M1-N5-C:J04-U01 2005-08-29-07.47.12.822993 R51-M1-N5-C:J04-U0

In [ ]:
# === RATE: fill in scores and run ===
rate(DATASET, IDX,
    correctness=0,           # 1-5
    completeness=0,          # 1-5
    evidence_grounding=0,    # 1-5
    actionable='',           # 'Y' or 'N'
    notes='',                # optional
)

[OK] BGL_03298600: C=5 Co=4 E=5 A=Y
     BGL progress: 1/50


## 5. Progress

Run this cell anytime to check how many sessions have been rated.

In [7]:
# === Progress Summary ===
print("=" * 50)
print("EVALUATION PROGRESS")
print("=" * 50)

total_rated = 0
for ds in ['BGL', 'HDFS']:
    sessions = prepared.get(ds, [])
    rated_sids = [s['session_id'] for s in sessions
                  if s['session_id'] in ratings_data['ratings']]
    unrated_idx = [i for i, s in enumerate(sessions)
                   if s['session_id'] not in ratings_data['ratings']]
    pct = len(rated_sids) / len(sessions) * 100 if sessions else 0
    total_rated += len(rated_sids)

    print(f"\n{ds}: {len(rated_sids)}/{len(sessions)} ({pct:.0f}%)")
    if unrated_idx:
        show = unrated_idx[:15]
        tail = f" ... +{len(unrated_idx)-15} more" if len(unrated_idx) > 15 else ""
        print(f"  Unrated indices: {show}{tail}")
    else:
        print(f"  [OK] ALL DONE")

    # Show signature coverage of rated sessions
    if rated_sids:
        rated_sigs = set(ratings_data['ratings'][sid]['normalized_signature']
                         for sid in rated_sids)
        all_sigs = set(s['normalized_signature'] for s in sessions)
        print(f"  Signatures rated: {len(rated_sigs)}/{len(all_sigs)}")

print(f"\n{'=' * 50}")
print(f"Total: {total_rated}/100 sessions rated")

EVALUATION PROGRESS

BGL: 1/50 (2%)
  Unrated indices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] ... +34 more
  Signatures rated: 1/50

HDFS: 0/50 (0%)
  Unrated indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14] ... +35 more

Total: 1/100 sessions rated


## 6. Analysis

Run this section **after all 100 sessions are rated**.
Produces summary statistics, per-dataset comparison, and paper-ready table + figure.

In [ ]:
# === Analysis ===
rows = []
for sid, r in ratings_data['ratings'].items():
    rows.append({
        'session_id': sid,
        'dataset': r['dataset'],
        'signature': r['normalized_signature'],
        'correctness': r['correctness'],
        'completeness': r['completeness'],
        'evidence_grounding': r['evidence_grounding'],
        'actionable': 1 if r['actionable'] == 'Y' else 0,
    })
df_r = pd.DataFrame(rows)
assert len(df_r) > 0, "No ratings found -- rate some sessions first"

DIMS = ['correctness', 'completeness', 'evidence_grounding']

# --- Overall ---
print("=" * 70)
print("HUMAN EVALUATION RESULTS")
print("=" * 70)
print(f"\nTotal rated: {len(df_r)} sessions")
print(f"\n{'Dimension':<25} {'Mean':>6} {'Median':>8} {'Std':>6} {'Min':>5} {'Max':>5}")
print("-" * 60)
for d in DIMS:
    v = df_r[d]
    print(f"{d:<25} {v.mean():>6.2f} {v.median():>8.1f} {v.std():>6.2f} "
          f"{v.min():>5} {v.max():>5}")
print(f"{'actionable':<25} {df_r['actionable'].mean():>6.1%}")

# --- By Dataset ---
print("\n--- By Dataset ---")
for ds in ['BGL', 'HDFS']:
    sub = df_r[df_r['dataset'] == ds]
    if sub.empty:
        continue
    print(f"\n{ds} (n={len(sub)}):")
    print(f"  {'Dimension':<25} {'Mean':>6} {'Std':>6}")
    for d in DIMS:
        print(f"  {d:<25} {sub[d].mean():>6.2f} {sub[d].std():>6.2f}")
    print(f"  {'actionable':<25} {sub['actionable'].mean():>6.1%}")

# --- By Signature (top 10) ---
print("\n--- By Signature (top 10 by sample count) ---")
sig_counts = df_r['signature'].value_counts().head(10)
for sig, cnt in sig_counts.items():
    sub = df_r[df_r['signature'] == sig]
    print(f"  {sig:<45} n={cnt:>2}  "
          f"corr={sub['correctness'].mean():.1f}  "
          f"comp={sub['completeness'].mean():.1f}  "
          f"evid={sub['evidence_grounding'].mean():.1f}")

# --- Correctness vs verification ---
print("\n--- Correctness vs Automated Verification ---")
for ds in ['BGL', 'HDFS']:
    sessions = prepared.get(ds, [])
    for s in sessions:
        sid = s['session_id']
        if sid in ratings_data['ratings']:
            idx_r = next(i for i, r in enumerate(rows) if r['session_id'] == sid)
            rows[idx_r]['verification_passed'] = s.get('verification_passed', None)
df_v = pd.DataFrame(rows)
if 'verification_passed' in df_v.columns:
    for vp_val, label in [(True, 'Verification PASSED'), (False, 'Verification FAILED')]:
        sub = df_v[df_v['verification_passed'] == vp_val]
        if not sub.empty:
            print(f"  {label} (n={len(sub)}): "
                  f"corr={sub['correctness'].mean():.2f}, "
                  f"evid={sub['evidence_grounding'].mean():.2f}")

In [ ]:
# === Paper-Ready Table ===
print("\n" + "=" * 90)
print("TABLE: Human Evaluation of Explanation Quality (Likert 1-5, n=50 per dataset)")
print("=" * 90)
header = (f"| {'Dataset':<8} | {'n':>3} | {'Correctness':>15} | {'Completeness':>15} "
          f"| {'Evid. Grounding':>17} | {'Actionable':>11} |")
sep    = f"|{'-'*10}|{'-'*5}|{'-'*17}|{'-'*17}|{'-'*19}|{'-'*13}|"
print(header)
print(sep)
for ds in ['BGL', 'HDFS', 'Overall']:
    sub = df_r if ds == 'Overall' else df_r[df_r['dataset'] == ds]
    if sub.empty:
        continue
    n = len(sub)
    def fmt(col):
        return f"{sub[col].mean():.2f} +/- {sub[col].std():.2f}"
    act = f"{sub['actionable'].mean():.0%}"
    print(f"| {ds:<8} | {n:>3} | {fmt('correctness'):>15} | {fmt('completeness'):>15} "
          f"| {fmt('evidence_grounding'):>17} | {act:>11} |")

# === Distribution Figure (grayscale, 600 dpi) ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, dim in zip(axes, DIMS):
    for ds, ls, gray, marker in [('BGL', '-', '0.1', 'o'), ('HDFS', '--', '0.5', 's')]:
        sub = df_r[df_r['dataset'] == ds]
        if sub.empty:
            continue
        counts = sub[dim].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
        ax.plot([1, 2, 3, 4, 5], counts.values, ls,
                color=gray, marker=marker, markersize=5, label=ds)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.set_title(dim.replace('_', ' ').title())
    ax.legend(frameon=False)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_ylim(bottom=0)

plt.tight_layout()
out_path = PROJECT_ROOT / 'results' / 'paper_human_eval_distribution.png'
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"[OK] Saved: {out_path.name}")